# 🎭 KoCulture-Dialogues SLM 파인튜닝 (QLoRA)

**한국 신조어 데이터셋으로 Qwen2.5-3B 모델을 파인튜닝합니다.**

## 실행 환경
- **권장**: Google Colab + T4 GPU (무료) 또는 A100 (Pro)
- **소요 시간**: T4에서 약 1.5~2시간 / A100에서 약 25분
- **VRAM**: 약 10GB 사용

## 진행 순서
1. GPU 확인 → 라이브러리 설치
2. 데이터셋 로드 및 전처리
3. 모델 로드 (4-bit 양자화)
4. **파인튜닝 전** 답변 테스트 (비교용)
5. LoRA 설정 → 학습
6. **파인튜닝 후** 답변 테스트
7. 어댑터 저장

> ⚠️ **시작 전 필수**: Colab에서 `런타임 > 런타임 유형 변경 > GPU > T4` 설정

## Step 0. GPU 환경 확인

먼저 GPU가 제대로 잡혔는지 확인합니다.

In [ ]:
!nvidia-smi

Fri May 15 07:02:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 1. 필수 라이브러리 설치

QLoRA 학습에 필요한 라이브러리들을 설치합니다. 버전 고정이 중요해요 — 안 그러면 호환성 문제가 자주 발생합니다.

In [ ]:
!pip install -q -U \
    transformers==4.46.0 \
    peft==0.13.2 \
    bitsandbytes==0.45.3 \
    trl==0.11.4 \
    datasets==3.0.0 \
    accelerate==1.0.1 \

print("✅ 설치 완료. 런타임 재시작이 필요할 수 있어요.")

✅ 설치 완료. 런타임 재시작이 필요할 수 있어요.


> ⚠️ 위 셀 실행 후 노란색 안내문이 뜨면 `런타임 > 세션 다시 시작`을 누르고 이 셀부터 다시 실행하세요. 안 떴으면 그냥 진행.

## Step 2. 라이브러리 import

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## Step 3. 데이터셋 로드 & 살펴보기

In [ ]:
from datasets.builder import VerificationMode
# KoCulture-Dialogues 로드 (10,356 행)
ds = load_dataset("huggingface-KREW/KoCulture-Dialogues", split="train", verification_mode=VerificationMode.NO_CHECKS)

print(f"전체 데이터 수: {len(ds)}")
print(f"필드: {ds.column_names}")
print()
print("=== 샘플 3개 ===")
for i in [0, 100, 5000]:
    print(f"\n[{ds[i]['title']}]")
    print(f"Q: {ds[i]['question']}")
    print(f"A: {ds[i]['answer']}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.28M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3456 [00:00<?, ? examples/s]

전체 데이터 수: 10356
필드: ['title', 'question', 'answer']

=== 샘플 3개 ===

[추구미]
Q: ㅋㅋㅋ 지수야 인스타 스토리 봤는데 또 카페 투어했네?
A: ㅇㅇ ㅋㅋㅋ 조용한 동네에 작은 로스터리 카페 사장님이 되는 게 내 추구미라... 요즘 열심히 리서치 중

[경찰서 정모]
Q: 팀장님 부르는데? 아까 회계팀이랑 마케팅팀 다 불려갔어 회의실로
A: 뭐야 설마 지난주 접대비 영수증 조작한거 걸린거야?? 담당자들 전부 경찰서 정모각인데... 난 몰랐다고 해야지 ㅠㅠ

[n트]
Q: 나 또 이번에 독서록 쓰다가 첫 줄부터 지움ㅋㅋㅋㅋ
A: 아니 너 매번 그러더라 ㅋㅋㅋ 한 5트까지 간 거 아님?


## Step 4. 데이터 전처리

원본은 `title/question/answer` 구조지만, 학습엔 대화 형식이 필요합니다.
`question`을 user 메시지로, `answer`를 assistant 메시지로 변환합니다.

In [ ]:
def to_chat_format(example):
    return {
        "messages": [
            {"role": "user", "content": example["question"]},
            {"role": "assistant", "content": example["answer"]},
        ]
    }

ds = ds.map(to_chat_format, remove_columns=ds.column_names)

# Train/Validation 9:1 분할
ds = ds.train_test_split(test_size=0.1, seed=42)

print(f"Train: {len(ds['train'])}")
print(f"Eval:  {len(ds['test'])}")
print()
print("=== 변환된 샘플 ===")
print(ds["train"][0])

Map:   0%|          | 0/10356 [00:00<?, ? examples/s]

Train: 9320
Eval:  1036

=== 변환된 샘플 ===
{'messages': [{'content': '야 가로쉬 이번에 너프되고 사람들 겉바속촉이라 부름ㅋㅋㅋㅋ', 'role': 'user'}, {'content': '진짜 겉바속촉ㅋㅋ 얘 플레이 자체가 튀김옷 얇은 통닭 같았잖아ㅋㅋ', 'role': 'assistant'}]}


## Step 5. 모델 & 토크나이저 로드 (4-bit 양자화)

Qwen2.5-3B-Instruct를 4-bit로 압축하여 메모리를 절약합니다.
원래 약 6GB짜리 모델이 약 2GB로 줄어듭니다.

In [ ]:
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
from peft import prepare_model_for_kbit_training
import torch

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# 4-bit 양자화 설정 (QLoRA의 'Q' 부분)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # Normal Float 4-bit
    bnb_4bit_compute_dtype=torch.float16, # T4 GPU는 float16에 더 최적화되어 있음
    bnb_4bit_use_double_quant=True,     # 양자화 상수도 양자화
)

# 토크나이저
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 모델 로드 (자동으로 GPU에 올림)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16, # T4 GPU는 float16에 더 최적화되어 있음
)

# LoRA 학습 준비 (gradient checkpointing 등)
model = prepare_model_for_kbit_training(model)

print(f"✅ 모델 로드 완료")
print(f"메모리 사용량: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ 모델 로드 완료
메모리 사용량: 2.68 GB


## Step 6. ⭐ 파인튜닝 **전** 답변 테스트

학습 시작 전에 베이스 모델이 신조어를 어떻게 다루는지 미리 보고 갑니다.
나중에 비교용 자료로 쓸 수 있어요 — 블로그에 들어갈 핵심 콘텐츠입니다.

In [ ]:
def chat(prompt, max_new_tokens=200):
    """모델에게 질문을 던지고 답변을 받습니다."""
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return response

# 평가용 질문 (학습 데이터에 없는 신조어 위주로!)
test_questions = [
    "친구가 게임에서 봉산탈춤 추고 있다는데 뭔 뜻이야?",
    "내 추구미는 미니멀한 인테리어인데 어떻게 꾸미면 좋을까?",
    "어제 콘서트 진짜 어마무시했어",
    "쟤 음주운전하다 경찰서 정모 갔대",
    "오늘 발표 폼 미쳤다",
]

print("=" * 70)
print("🎯 파인튜닝 BEFORE")
print("=" * 70)
before_results = {}
for q in test_questions:
    answer = chat(q)
    before_results[q] = answer
    print(f"\nQ: {q}")
    print(f"A: {answer}")
    print("-" * 70)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


🎯 파인튜닝 BEFORE

Q: 친구가 게임에서 봉산탈춤 추고 있다는데 뭔 뜻이야?
A: 봉산탈춤은 한국의 전통 춤추기로, 서울특별시 봉산동에서 출전하는 지역별 대회에서 선정되는 대표적인 춤무지입니다. 하지만, 현재 게임에서 이런 이름의 춤추기가 나오는 것은 상식적으로 불가능합니다.

게임 내에서 "봉산탈춤"이라는 이름을 가진 캐릭터나 아이템, 혹은 무기를 만나게 되면, 그 설정에 따라 그 의미를 찾아볼 수 있습니다. 게임에 대한 자세한 정보가 더 필요합니다. 아니면 실제 춤추기의 의미를 말씀해주시면 도와드리겠습니다.
----------------------------------------------------------------------

Q: 내 추구미는 미니멀한 인테리어인데 어떻게 꾸미면 좋을까?
A: 미니멀한 인테리어를 위해 꾸미는 방법에 대해 몇 가지 제안을 드립니다:

1. 간결한 색상 선택: 베이지, 노랑, 흰색 등 단조로운 색상을 선택하세요.

2. 적절한 장식: 과도한 장식을 피하고, 필요한 것만 사용하세요.

3. 원목 소재: 자연과 더近합하는 원목 소재를 활용하면 좋습니다.

4. 소파: 뒤집어져 있는 모양의 소파를 선택하면 공간이 넓어 보입니다.

5. 대형 가구: 하나의 큰 가구를 사용하여 공간을 차지합니다.

6. 가독성 좋은 책장: 책 표지가 보이도록 하는 책장으로 변경하세요.

7. 개인화 요소: 특별한 개인적인 것, 예를 들면
----------------------------------------------------------------------

Q: 어제 콘서트 진짜 어마무시했어
A: 그런 높은 평가에 감사드립니다! 콘서트가 정말로 멋지셨나 봅니다. 어떤 부분이 특히 기억나신 건가요? 저는 항상 고객님의 경험을 더욱 특별하게 만들어드릴 수 있도록 최선을 다하고 있습니다. 혹시 추가로 도움이 필요하시다면 알려주세요!
----------------------------------------------------------

## Step 7. LoRA 설정

학습할 LoRA 어댑터를 정의합니다. 핵심 하이퍼파라미터:
- `r=16`: LoRA 랭크. 클수록 표현력↑, 메모리↑
- `lora_alpha=32`: 스케일링 계수 (보통 r의 2배)
- `target_modules`: LoRA를 적용할 레이어들

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

print("✅ LoRA 설정 완료")

✅ LoRA 설정 완료


## Step 8. 학습 설정

SFTConfig로 학습 하이퍼파라미터를 정의합니다. T4에서는 batch_size=2 권장,
A100이면 batch_size=8까지 올려도 OK.

In [ ]:
sft_config = SFTConfig(
    # 출력 경로
    output_dir="./output/koculture-lora",

    # 학습 길이
    num_train_epochs=3,
    per_device_train_batch_size=2,       # T4 기준. A100이면 8까지 OK
    gradient_accumulation_steps=4,        # 실효 배치 = 2 × 4 = 8

    # 학습률
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    # 메모리 최적화
    bf16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,

    # 로깅 & 저장
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=2,

    # SFT 전용
    max_seq_length=512,
    packing=False,

    # 기타
    report_to="none",                    # wandb 등 비활성화
    seed=42,
)

print("✅ 학습 설정 완료")

✅ 학습 설정 완료


## Step 9. 🚀 학습 실행!

SFTTrainer가 LoRA 적용 + 학습을 자동으로 처리합니다.
T4 기준 약 1.5~2시간 걸려요. 진행 상황은 loss로 확인.

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    peft_config=lora_config,
    tokenizer=tokenizer,
)

# 학습 시작
trainer.train()

print("\n✅ 학습 완료!")

Map:   0%|          | 0/9320 [00:00<?, ? examples/s]

## Step 10. 어댑터 저장

학습된 LoRA 어댑터만 저장합니다 (~60MB).

In [ ]:
SAVE_PATH = "./output/koculture-lora-final"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

# Colab에서 Google Drive에 백업하고 싶다면:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r ./output/koculture-lora-final /content/drive/MyDrive/

print(f"✅ 저장 완료: {SAVE_PATH}")
!du -sh {SAVE_PATH}

## Step 11. ⭐ 파인튜닝 **후** 답변 테스트

같은 질문을 학습된 모델에게 던져봅니다. 위에서 봤던 Before와 비교해보세요!

In [ ]:
print("=" * 70)
print("🎯 파인튜닝 AFTER")
print("=" * 70)

after_results = {}
for q in test_questions:
    answer = chat(q)
    after_results[q] = answer
    print(f"\nQ: {q}")
    print(f"A: {answer}")
    print("-" * 70)

## Step 12. Before vs After 비교 표

블로그에 그대로 넣을 수 있는 비교 표를 만듭니다.

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "질문": test_questions,
    "파인튜닝 전": [before_results[q] for q in test_questions],
    "파인튜닝 후": [after_results[q] for q in test_questions],
})

# 보기 좋게 출력
pd.set_option("display.max_colwidth", None)
display(comparison)

# CSV로 저장 (블로그용)
comparison.to_csv("./output/before_after_comparison.csv", index=False, encoding="utf-8-sig")
print("\n✅ CSV 저장 완료: ./output/before_after_comparison.csv")

## Step 13. (참고) 나중에 어댑터만 불러와서 쓰기

학습이 다 끝난 후, 어댑터만 따로 불러와서 추론하는 방법입니다.
블로그의 `inference.py` 데모 코드로 활용 가능.

In [ ]:
# 🔁 새 세션에서는 이 코드만 실행하면 됩니다

# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# from peft import PeftModel
# import torch
#
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
# )
#
# base = AutoModelForCausalLM.from_pretrained(
#     "Qwen/Qwen2.5-3B-Instruct",
#     quantization_config=bnb_config,
#     device_map="auto",
# )
# tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
# model = PeftModel.from_pretrained(base, "./output/koculture-lora-final")
# model.eval()

print("위 코드는 새 세션에서 어댑터만 불러올 때 사용합니다.")

## ✅ 다 끝났습니다!

### 다음 할 일

1. **결과를 README에 반영**
   - Step 12에서 만든 비교 표를 블로그 IV-1 섹션에 붙여넣기
   - Loss curve는 `trainer.state.log_history`에서 추출해서 matplotlib으로 그리기
   - 효율성 지표 (학습 시간, GPU 메모리, 어댑터 크기) 측정해서 IV-3 섹션에 기록

2. **추가 평가**
   - BLEU/ROUGE 자동 평가 (별도 노트북에서)
   - 팀원끼리 사람 평가

3. **모델 공유 (선택)**
   - `trainer.push_to_hub("your-username/koculture-qwen2.5-3b-lora")` 로 HF에 업로드
   - README에 모델 링크 첨부하면 후한 평가 받을 가능성↑

### 자주 발생하는 에러 대처

| 에러 메시지 | 해결 |
|------------|------|
| `CUDA out of memory` | `per_device_train_batch_size`를 1로 줄이고 `gradient_accumulation_steps`를 8로 |
| `bitsandbytes` import 실패 | 런타임 재시작 후 다시 import |
| 학습이 너무 느림 | Colab Pro로 A100 GPU 할당받기 |
| 답변이 영어로 나옴 | `temperature`를 0.5로 낮춰보기 |

화이팅! 🚀